In [1]:
import quante as qt

In [2]:
# 生成具有粒子数生活和动量守恒的基矢
basis = qt.generate.basis.spin_basis(L=10, Nup=5, kblock=1)

# 可以查看基矢的个数：
print(basis.Ns)

# 可以获得某个基矢在全空间中的表示
state = basis.to_full_space(0)

# 可以可视化全空间的基矢：
qt.generate.basis.show_spin_basis(state)

25
↑↑↑↑↑↓↓↓↓↓: [0.31622777+0.j]
↑↑↑↑↓↓↓↓↓↑: [0.25583364+0.18587402j]
↑↑↑↓↓↓↓↓↑↑: [0.09771975+0.30075048j]
↑↑↓↓↓↓↓↑↑↑: [-0.09771975+0.30075048j]
↑↓↓↓↓↓↑↑↑↑: [-0.25583364+0.18587402j]
↓↑↑↑↑↑↓↓↓↓: [0.25583364-0.18587402j]
↓↓↑↑↑↑↑↓↓↓: [0.09771975-0.30075048j]
↓↓↓↑↑↑↑↑↓↓: [-0.09771975-0.30075048j]
↓↓↓↓↑↑↑↑↑↓: [-0.25583364-0.18587402j]
↓↓↓↓↓↑↑↑↑↑: [-0.31622777-0.j]


In [3]:
# 定义哈密顿量：
op = qt.generate.operas
L = 10
ham = op.sum(op.xx(i,(i+1)%L) + op.yy(i,(i+1)%L) + 0.5 * op.zz(i,(i+1)%L) for i in range(L-1))

ham.show_string_form()

xx
  (0, 1) 1.0
  (1, 2) 1.0
  (2, 3) 1.0
  (3, 4) 1.0
  (4, 5) 1.0
  (5, 6) 1.0
  (6, 7) 1.0
  (7, 8) 1.0
  (8, 9) 1.0
yy
  (0, 1) 1.0
  (1, 2) 1.0
  (2, 3) 1.0
  (3, 4) 1.0
  (4, 5) 1.0
  (5, 6) 1.0
  (6, 7) 1.0
  (7, 8) 1.0
  (8, 9) 1.0
zz
  (0, 1) 0.5
  (1, 2) 0.5
  (2, 3) 0.5
  (3, 4) 0.5
  (4, 5) 0.5
  (5, 6) 0.5
  (6, 7) 0.5
  (7, 8) 0.5
  (8, 9) 0.5



In [4]:
# 获得哈密顿量在给定基矢下的矩阵：
mat = ham.to_matrix(basis)
mat

array([[ 0.875    +0.j        ,  0.5      +0.j        ,  0.       +0.j        ,  0.       +0.j        ,  0.4045085-0.29389263j,  0.       +0.j        ,  0.       +0.j        ,  0.       +0.j        ,  0.       +0.j        ,  0.       +0.j        ,  0.       +0.j        ,  0.       +0.j        ,  0.       +0.j        ,  0.       +0.j        ,  0.       +0.j        ,  0.       +0.j        ,  0.       +0.j        ,  0.       +0.j        ,  0.       +0.j        ,  0.       +0.j        ,  0.       +0.j        ,  0.       +0.j        ,  0.       +0.j        ,  0.       +0.j        ,  0.       +0.j        ],
       [ 0.5      +0.j        ,  0.375    +0.j        ,  0.5      +0.j        ,  0.       +0.j        ,  0.       +0.j        ,  0.5      +0.j        ,  0.       +0.j        ,  0.       +0.j        ,  0.4045085-0.29389263j,  0.       +0.j        ,  0.       +0.j        ,  0.       +0.j        ,  0.       +0.j        ,  0.       +0.j        ,  0.       +0.j        ,  0.       +0.j        ,

In [13]:
# 对角化
engs, eigstates = qt.linalg.eigh(mat, k=1)  # 获得最低能量的本征态
engs

array([-2.3894809])

In [14]:
# 计算纠缠：
entspect = qt.quantity.entanglement_spectrum(eigstates[:,0], L, L//2, basis)  # 纠缠谱
qt.quantity.entropy(entspect)

array(4.00114096)

In [1]:
# 对比 quspin 和 quante 的效率（需要在安装 quspin 的环境中运行）
import sys
sys.dont_write_bytecode = True

import quante as qt
import numpy as np
import time

from quspin.basis import spin_basis_1d
from quspin.operators import hamiltonian

L = 20
ham = qt.generate.operas.heisenberg_operator(L, j=(1, 1, 1))
ham = ham.expandxy(pauli=False)
quspin_basis = spin_basis_1d(L=L, pauli=0)
lis = ham.quspin_form()

t = time.time()
mat3 = hamiltonian(lis, [], basis=quspin_basis, check_symm=False, check_herm=False, dtype=np.float64).static
print("quspin time: ", time.time()-t)

basis = qt.generate.basis.spin_basis(L=L)
t = time.time()
mat1 = ham.to_matrix(basis, sparse=True)
print("quante time: ", time.time()-t)


print("diff: ",qt.linalg.norm(mat1 - mat3))

quspin time:  2.1174919605255127
quante time:  0.504335880279541
diff:  0.0


In [4]:
# 没有对称性时,快速生成矩阵元
# 方法1 正常 cpu 方法
import time
import quante as qt

basis = qt.generate.basis.spin_basis(24)
ham = qt.generate.operas.heisenberg_operator(24)
ham = ham.expandxy(False)

t = time.time()
ham.to_matrix(basis, sparse=True)
print("time:", time.time()-t)

time: 8.732816457748413


In [2]:
# 方法2 MPO 收缩方法
import time
import quante as qt
from quante.tensor.automata import get_sparse_matrix

basis = qt.generate.basis.spin_basis(24)
ham = qt.generate.operas.heisenberg_operator(24)
ham = ham.expandxy(False)

t = time.time()
mat = get_sparse_matrix(24, *ham.split_data(), pauli=False)
print("time:", time.time()-t)

time: 7.106258392333984


In [4]:
# 使用 GPU 加速
import time
import quante as qt
from quante.torch_utils.symmetry.basis import to_matrix_cuda

basis = qt.generate.basis.spin_basis(24)
ham = qt.generate.operas.heisenberg_operator(24)
eachterm, hascomplex = ham.expandxy(False)._convert_to_quick_form()

t = time.time()
mat = to_matrix_cuda(basis, eachterm, hascomplex)
print("time:", time.time()-t)

time: 4.179324388504028


In [2]:
# 方法2 GPU + MPO 收缩方法
import time
import quante as qt
from quante.tensor.automata import get_sparse_matrix

basis = qt.generate.basis.spin_basis(24)
ham = qt.generate.operas.heisenberg_operator(24)
ham = ham.expandxy(False)

t = time.time()
mat = get_sparse_matrix(24, *ham.split_data(), pauli=False, usecuda=True)
print("time:", time.time()-t)

time: 3.01497220993042


生成矩阵的难点在于, 稀疏矩阵的加法, 它无法利用并行加速

automata 之所以更快是因为, 它最小化了大型稀疏矩阵加法的次数

In [ ]:
# 验证程序

basis = qt.generate.basis.spin_basis(10)
ham = qt.generate.operas.heisenberg_operator(10)
ham = ham.expandxy(False)
mat1 = ham.to_matrix(basis)

oplist, hascomplex = ham.expandxy(False)._convert_to_quick_form()
mat2 = to_matrix_cuda(basis, oplist, hascomplex)

print(np.allclose(mat1, mat2.to_dense().cpu().numpy()))